# PPO Demo on the Crawler

This notebook keeps the **same crawler environment**, **same 10-seed setup**, and **same 600-episode experiment budget** used in Demo 7 of `L6-2_demo_crawler_pg.ipynb`.

Main message:
- On the current saved crawler outputs, PPO does **not** beat the best baseline variants in raw reward; the strongest crawler baseline remains higher.
- PPO still shows the intended algorithmic effect: its raw advantage-variability proxy is much lower than the REINFORCE / baseline variants on this task, which is consistent with **better-conditioned updates**.
- So the crawler section is best read as a bridge: it shows why PPO is attractive, but the clearer scaling story comes from the **Humanoid 3D** section later in this notebook.
- To make the comparison fair and fast, we **load the previously saved REINFORCE / baseline / Actor-Critic checkpoints** whenever they are available, and only train PPO if its checkpoints are missing.

In [ ]:
# Setup
import numpy as np
import gymnasium as gym
import mujoco
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from pathlib import Path
from io import BytesIO
import base64
import time
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

from stable_baselines3 import PPO as SB3PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecNormalize

print(f'MuJoCo version: {mujoco.__version__}')
print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Setup complete!')


In [ ]:
# ============================================================
# Crawler Environment (same as demo_crawler_rl.ipynb)
# ============================================================

CRAWLER_XML = """
<mujoco model="crawler2d">
  <compiler angle="degree" inertiafromgeom="true"/>
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4"/>

  <default>
    <geom conaffinity="1" condim="3" friction="1.5 0.5 0.1" density="1000"/>
    <joint armature="0.1" damping="0.5"/>
  </default>

  <asset>
    <texture type="2d" name="grid" builtin="checker" width="512" height="512"
             rgb1="0.7 0.9 0.7" rgb2="0.6 0.85 0.6"/>
    <material name="grid" texture="grid" texrepeat="8 8"/>
  </asset>

  <worldbody>
    <light diffuse="0.8 0.8 0.8" pos="0 -2 3" dir="0 0.5 -1"/>
    <geom name="floor" type="plane" size="50 1 0.1" material="grid"/>

    <geom name="origin" type="box" size="0.01 0.15 0.002" pos="0 0 0.001" rgba="0.1 0.1 0.8 0.7" contype="0" conaffinity="0"/>
    <geom name="ruler_m4" type="box" size="0.006 0.12 0.002" pos="-2.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_m3" type="box" size="0.003 0.08 0.001" pos="-1.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_m2" type="box" size="0.006 0.12 0.002" pos="-1.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_m1" type="box" size="0.003 0.08 0.001" pos="-0.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_1" type="box" size="0.003 0.08 0.001" pos="0.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_2" type="box" size="0.006 0.12 0.002" pos="1.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_3" type="box" size="0.003 0.08 0.001" pos="1.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_4" type="box" size="0.006 0.12 0.002" pos="2.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_5" type="box" size="0.003 0.08 0.001" pos="2.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_6" type="box" size="0.006 0.12 0.002" pos="3.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_7" type="box" size="0.003 0.08 0.001" pos="3.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_8" type="box" size="0.006 0.12 0.002" pos="4.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_9" type="box" size="0.003 0.08 0.001" pos="4.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_10" type="box" size="0.006 0.12 0.002" pos="5.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_11" type="box" size="0.003 0.08 0.001" pos="5.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_12" type="box" size="0.006 0.12 0.002" pos="6.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_13" type="box" size="0.003 0.08 0.001" pos="6.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_14" type="box" size="0.006 0.12 0.002" pos="7.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_15" type="box" size="0.003 0.08 0.001" pos="7.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_16" type="box" size="0.006 0.12 0.002" pos="8.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_17" type="box" size="0.003 0.08 0.001" pos="8.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_18" type="box" size="0.006 0.12 0.002" pos="9.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>
    <geom name="ruler_19" type="box" size="0.003 0.08 0.001" pos="9.5 0 0.001" rgba="0.25 0.25 0.25 0.4" contype="0" conaffinity="0"/>
    <geom name="ruler_20" type="box" size="0.006 0.12 0.002" pos="10.0 0 0.001" rgba="0.15 0.15 0.15 0.6" contype="0" conaffinity="0"/>

    <camera name="side" pos="0 -0.8 0.25" xyaxes="1 0 0 0 0.3 1" mode="trackcom"/>

    <body name="torso" pos="0 0 0.035">
      <joint name="root_x" type="slide" axis="1 0 0"/>
      <joint name="root_z" type="slide" axis="0 0 1"/>
      <joint name="root_rot" type="hinge" axis="0 1 0"/>

      <geom name="torso_geom" type="box" size="0.08 0.035 0.025"
            rgba="0.3 0.75 0.3 1" density="3000"/>

      <body name="arm" pos="0.08 0 0.01">
        <joint name="arm_joint" type="hinge" axis="0 1 0"
               range="-70 70" limited="true"/>
        <geom name="arm_geom" type="capsule" size="0.012"
              fromto="0 0 0 0.12 0 0" rgba="0.95 0.7 0.1 1"/>

        <body name="hand" pos="0.12 0 0">
          <joint name="hand_joint" type="hinge" axis="0 1 0"
                 range="-70 70" limited="true"/>
          <geom name="hand_geom" type="capsule" size="0.008"
                fromto="0 0 0 0.08 0 0" rgba="0.9 0.15 0.15 1"/>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="arm_motor" joint="arm_joint" ctrllimited="true"
           ctrlrange="-1 1" gear="5"/>
    <motor name="hand_motor" joint="hand_joint" ctrllimited="true"
           ctrlrange="-1 1" gear="3"/>
  </actuator>
</mujoco>
"""


class CrawlerEnv:
    """2D MuJoCo crawler with 2 actuated joints.

    State:  4D continuous (arm angle, hand angle, arm vel, hand vel)
    Action: 2D continuous torque in [-1, 1]
    Reward: forward (x) velocity
    """

    def __init__(self, max_steps=500, frame_skip=4):
        self.model = mujoco.MjModel.from_xml_string(CRAWLER_XML)
        self.data = mujoco.MjData(self.model)
        self.max_steps = max_steps
        self.frame_skip = frame_skip
        self.steps = 0
        self.obs_dim = 4
        self.act_dim = 2

    def get_obs(self):
        arm_a, hand_a = self.data.qpos[3], self.data.qpos[4]
        arm_v, hand_v = self.data.qvel[3], self.data.qvel[4]
        return np.array([arm_a, hand_a, arm_v, hand_v], dtype=np.float32)

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[3] = np.random.uniform(-0.1, 0.1)
        self.data.qpos[4] = np.random.uniform(-0.1, 0.1)
        mujoco.mj_forward(self.model, self.data)
        self.steps = 0
        return self.get_obs()

    def step(self, ctrl):
        x_before = self.data.qpos[0]
        self.data.ctrl[:] = np.clip(ctrl, -1, 1)
        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)
        x_after = self.data.qpos[0]
        dt = self.frame_skip * self.model.opt.timestep
        reward = (x_after - x_before) / dt
        self.steps += 1
        truncated = self.steps >= self.max_steps
        return self.get_obs(), reward, False, truncated, {'x': x_after}


print(f'Environment: obs_dim={CrawlerEnv().obs_dim}, act_dim={CrawlerEnv().act_dim}')
print('Actions are now CONTINUOUS 2D torques in [-1, 1] -- no more discrete action table!')

In [ ]:
# ============================================================
# Visualization helpers (same as previous notebook)
# ============================================================

eval_results = {}


def should_skip_rendering():
    """Best-effort detection of headless / remote sessions where OpenGL rendering is likely unavailable."""
    remote_markers = ('SSH_CONNECTION', 'SSH_CLIENT', 'SSH_TTY')
    is_remote = any(os.environ.get(name) for name in remote_markers)
    has_display = bool(os.environ.get('DISPLAY') or os.environ.get('WAYLAND_DISPLAY'))
    return is_remote and not has_display


def rollout_episode(env, policy_fn, max_steps=500):
    """Roll out a policy without rendering. Returns: (distance, total_reward)."""
    obs = env.reset()
    total_reward = 0.0
    for _ in range(max_steps):
        ctrl = policy_fn(obs)
        obs, reward, terminated, truncated, _ = env.step(ctrl)
        total_reward += reward
        if terminated or truncated:
            break
    dist = float(env.data.qpos[0])
    return dist, total_reward


def render_episode(env, policy_fn, max_steps=500, cam_name='side'):
    """Roll out a policy and collect rendered frames when OpenGL is available."""
    if should_skip_rendering():
        dist, total_reward = rollout_episode(env, policy_fn, max_steps=max_steps)
        print('Skipping video rendering: remote session without DISPLAY / Wayland detected.')
        return None, dist, total_reward

    try:
        renderer = mujoco.Renderer(env.model, height=320, width=560)
    except Exception as err:
        dist, total_reward = rollout_episode(env, policy_fn, max_steps=max_steps)
        print(f'Skipping video rendering: could not create OpenGL renderer ({type(err).__name__}: {err}).')
        return None, dist, total_reward

    frames = []
    obs = env.reset()
    total_reward = 0.0
    for _ in range(max_steps):
        renderer.update_scene(env.data, camera=cam_name)
        frames.append(renderer.render().copy())
        ctrl = policy_fn(obs)
        obs, reward, terminated, truncated, _ = env.step(ctrl)
        total_reward += reward
        if terminated or truncated:
            break
    dist = float(env.data.qpos[0])
    renderer.close()
    return frames, dist, total_reward


def eval_policy(env, policy_fn, label, max_steps=500):
    """Run a 10s rollout, render video, print distance, store result."""
    frames, dist, total_reward = render_episode(env, policy_fn, max_steps=max_steps)
    print(f'{label}: traveled {dist:.2f}m in 10s  (episode reward: {total_reward:.1f})')
    eval_results[label] = dist
    return frames, dist, total_reward


def show_video(frames, fps=30, title=None):
    if frames is None or len(frames) == 0:
        msg = 'Video skipped: no OpenGL context available in this session.'
        if title:
            msg = f'{title}<br><br>{msg}'
        return HTML(f"<div style='padding:1rem;border:1px solid #ccc;border-radius:8px'>{msg}</div>")

    fig, ax = plt.subplots(figsize=(8, 4))
    if title:
        ax.set_title(title, fontsize=14)
    ax.axis('off')
    im = ax.imshow(frames[0])
    def update(i):
        im.set_data(frames[i])
        return [im]
    anim = animation.FuncAnimation(fig, update, frames=len(frames),
                                   interval=1000/fps, blit=True)
    plt.close()
    return HTML(anim.to_html5_video())


def plot_rewards(rewards, window=50, title='Training Progress', ax=None, color='steelblue', label=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(rewards, alpha=0.15, color=color)
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color=color, linewidth=2, label=label)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    return ax


def plot_eval_comparison(results_dict, title='Policy Comparison: Distance Traveled in 10s'):
    labels = list(results_dict.keys())
    dists = list(results_dict.values())
    colors = ['#d9534f' if d < 0.5 else '#5bc0de' if d < 1.5 else '#5cb85c' for d in dists]

    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.5 + 1)))
    bars = ax.barh(labels, dists, color=colors, edgecolor='white', height=0.6)
    ax.set_xlabel('Distance (m)')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='gray', linewidth=0.5)
    for bar, d in zip(bars, dists):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{d:.2f}m', va='center', fontsize=11)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def stack_histories(histories):
    arrays = [np.asarray(h, dtype=np.float32) for h in histories if len(h) > 0]
    if not arrays:
        return np.zeros((0, 0), dtype=np.float32)
    min_len = min(len(h) for h in arrays)
    return np.stack([h[:min_len] for h in arrays], axis=0)


def smooth_histories(histories, window=40):
    data = stack_histories(histories)
    if data.size == 0:
        return np.array([], dtype=np.int32), np.zeros((0, 0), dtype=np.float32)
    if data.shape[1] < window:
        return np.arange(data.shape[1]), data
    smoothed = np.array([
        np.convolve(h, np.ones(window)/window, mode='valid')
        for h in data
    ])
    x = np.arange(window - 1, data.shape[1])
    return x, smoothed


def plot_seed_average(ax, histories, *, color, label, window=40, alpha=0.15):
    x, smoothed = smooth_histories(histories, window=window)
    if smoothed.size == 0:
        return
    mean = smoothed.mean(axis=0)
    std = smoothed.std(axis=0)
    ax.plot(x, mean, color=color, linewidth=2, label=label)
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=alpha)


def summarize_final_window(histories, tail=50):
    values = [float(np.mean(np.asarray(h)[-tail:])) for h in histories if len(h) > 0]
    if not values:
        return 0.0, 0.0
    return float(np.mean(values)), float(np.std(values))


# ============================================================
# Checkpoint helpers
# ============================================================

CHECKPOINT_DIR = Path('saved_checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)
LATEST_CHECKPOINT_PATH = CHECKPOINT_DIR / 'latest_pg_training.pt'

# Manually set this to True if you want to ignore cached files and retrain.
FORCE_RETRAIN = False


def checkpoint_slug(kind, meta):
    parts = [kind]
    for key in ('demo', 'variant', 'baseline', 'config_name', 'seed', 'n_episodes'):
        if key in meta and meta[key] is not None:
            parts.append(str(meta[key]))
    raw = '_'.join(parts)
    return ''.join(ch if ch.isalnum() or ch in '._-' else '_' for ch in raw)


def checkpoint_path_for(kind, meta):
    return CHECKPOINT_DIR / f'{checkpoint_slug(kind, meta)}.pt'


def save_pg_checkpoint(payload):
    payload = dict(payload)
    payload['saved_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    meta = payload.get('meta', {})
    path = checkpoint_path_for(payload.get('kind', 'pg'), meta)
    torch.save(payload, path)
    torch.save(payload, LATEST_CHECKPOINT_PATH)
    print(f'Saved checkpoint -> {path}')
    print(f'Updated latest checkpoint -> {LATEST_CHECKPOINT_PATH}')
    return path


def load_pg_checkpoint(path, verbose=True):
    if not path.exists():
        return None
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if verbose:
        print(f'Found checkpoint <- {path}')
        print(f"  label={payload.get('label', 'unknown')} | kind={payload.get('kind', 'unknown')}")
    return payload


def checkpoint_matches(payload, *, kind, required_meta, ignore_keys=()):
    if payload is None or payload.get('kind') != kind:
        return False
    meta = payload.get('meta', {})
    ignored = set(ignore_keys)
    return all(meta.get(k) == v for k, v in required_meta.items() if k not in ignored)


def find_compatible_pg_checkpoint(kind, required_meta, *, ignore_keys=('n_episodes',)):
    candidates = []
    for path in sorted(CHECKPOINT_DIR.glob(f'{kind}*.pt')):
        if path == LATEST_CHECKPOINT_PATH:
            continue
        payload = load_pg_checkpoint(path, verbose=False)
        if not checkpoint_matches(payload, kind=kind, required_meta=required_meta,
                                 ignore_keys=ignore_keys):
            continue
        meta = payload.get('meta', {})
        n_episodes = int(meta.get('n_episodes', -1) or -1)
        candidates.append((n_episodes, path, payload))

    if not candidates:
        return None, None

    candidates.sort(key=lambda item: (item[0], str(item[1])))
    _, path, payload = candidates[-1]
    return payload, path


def maybe_load_pg_checkpoint(*, kind, obs_dim, act_dim, hidden=64, label,
                             extra_meta=None, with_critic=False,
                             allow_compatible=True):
    if FORCE_RETRAIN:
        print(f'FORCE_RETRAIN=True for {label}; ignoring saved checkpoint.')
        return None

    required_meta = {
        'obs_dim': int(obs_dim),
        'act_dim': int(act_dim),
        'hidden': int(hidden),
    }
    if extra_meta:
        required_meta.update(extra_meta)

    matched_path = None
    specific_path = checkpoint_path_for(kind, required_meta)
    payload = load_pg_checkpoint(specific_path, verbose=False)
    if checkpoint_matches(payload, kind=kind, required_meta=required_meta):
        matched_path = specific_path
    else:
        payload = load_pg_checkpoint(LATEST_CHECKPOINT_PATH, verbose=False)
        if checkpoint_matches(payload, kind=kind, required_meta=required_meta):
            matched_path = LATEST_CHECKPOINT_PATH

    if matched_path is None and 'n_episodes' in required_meta and allow_compatible:
        payload, matched_path = find_compatible_pg_checkpoint(kind, required_meta)
        if payload is not None:
            saved_episodes = payload.get('meta', {}).get('n_episodes', 'unknown')
            print(
                f"Loaded compatible checkpoint for {label} from {matched_path} "
                f"(saved n_episodes={saved_episodes}, requested {required_meta['n_episodes']})."
            )
    elif matched_path is None and 'n_episodes' in required_meta and not allow_compatible:
        compatible_payload, compatible_path = find_compatible_pg_checkpoint(kind, required_meta)
        if compatible_payload is not None:
            saved_episodes = compatible_payload.get('meta', {}).get('n_episodes', 'unknown')
            print(
                f"Found only a mismatched checkpoint for {label} at {compatible_path} "
                f"(saved n_episodes={saved_episodes}, requested {required_meta['n_episodes']}); retraining."
            )

    if matched_path is None:
        return None

    policy = GaussianPolicy(obs_dim, act_dim, hidden).to(device)
    policy.load_state_dict(payload['policy_state_dict'])
    policy.eval()

    rewards = list(payload.get('rewards', []))
    extras = payload.get('extras', {})
    print(f'Loaded checkpoint for {label}; skipping retraining.')
    print(f'  path: {matched_path}')

    if with_critic:
        critic_hidden = int(required_meta.get('critic_hidden', hidden))
        critic = ValueNetwork(obs_dim, critic_hidden).to(device)
        critic.load_state_dict(payload['critic_state_dict'])
        critic.eval()
        return policy, critic, rewards, extras

    return policy, rewards, extras


In [ ]:
# ============================================================
# REINFORCE (Vanilla Policy Gradient)
# ============================================================

class GaussianPolicy(nn.Module):
    """Policy network that outputs mean of a Gaussian. std is a learnable parameter."""
    def __init__(self, obs_dim, act_dim, hidden=64, init_log_std=-0.7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim), nn.Tanh()  # output in [-1, 1]
        )
        # Learnable log-std for exploration scale.
        self.log_std = nn.Parameter(torch.full((act_dim,), float(init_log_std)))

    def forward(self, obs):
        mu = self.net(obs)
        std = self.log_std.exp()
        return mu, std

    def get_action(self, obs):
        """Sample an action and return (action, log_prob)."""
        mu, std = self.forward(obs)
        dist = Normal(mu, std)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        return action.clamp(-1, 1), log_prob

    def evaluate(self, obs, actions):
        """Compute log_prob of given actions (for batched computation)."""
        mu, std = self.forward(obs)
        dist = Normal(mu, std)
        return dist.log_prob(actions).sum(dim=-1)


def train_reinforce(env, n_episodes=500, gamma=0.99, lr=1e-3, hidden=64,
                    verbose=True, seed=None, checkpoint_label=None,
                    checkpoint_meta=None):
    """Train REINFORCE. Returns (policy, rewards_list)."""
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    policy = GaussianPolicy(env.obs_dim, env.act_dim, hidden).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    rewards_history = []
    adv_std_history = []

    t0 = time.time()
    for ep in range(n_episodes):
        obs = env.reset()
        log_probs = []
        rewards = []

        while True:
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            action, log_prob = policy.get_action(obs_t)
            act_np = action.squeeze(0).cpu().detach().numpy()

            next_obs, reward, terminated, truncated, _ = env.step(act_np)
            log_probs.append(log_prob)
            rewards.append(reward)
            obs = next_obs
            if terminated or truncated:
                break

        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        adv_std_history.append(float(returns.std().item()) if len(returns) > 1 else 0.0)

        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        log_probs_t = torch.stack(log_probs).squeeze()
        loss = -(log_probs_t * returns).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_reward = sum(rewards)
        rewards_history.append(total_reward)

        if verbose and (ep + 1) % 100 == 0:
            avg = np.mean(rewards_history[-50:])
            std_val = policy.log_std.exp().mean().item()
            print(f'  Episode {ep+1:4d} | Avg reward: {avg:7.1f} | std: {std_val:.3f}')

    elapsed = time.time() - t0
    if verbose:
        print(f'  Training completed in {elapsed:.1f}s')

    if checkpoint_label is not None:
        meta = {
            'obs_dim': int(env.obs_dim),
            'act_dim': int(env.act_dim),
            'hidden': int(hidden),
        }
        if checkpoint_meta:
            meta.update(checkpoint_meta)
        save_pg_checkpoint({
            'kind': 'reinforce',
            'label': checkpoint_label,
            'meta': meta,
            'policy_state_dict': policy.state_dict(),
            'rewards': np.asarray(rewards_history, dtype=np.float32),
            'extras': {
                'adv_std_history': np.asarray(adv_std_history, dtype=np.float32),
            },
        })

    return policy, rewards_history


In [ ]:
# ============================================================
# REINFORCE with Baseline (Actor-Critic)
# ============================================================

class ValueNetwork(nn.Module):
    """Critic: estimates V(s)."""
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, obs):
        return self.net(obs).squeeze(-1)


def train_actor_critic(env, n_episodes=500, gamma=0.99, lr_actor=1e-3,
                       lr_critic=1e-3, hidden=64, verbose=True, seed=None,
                       checkpoint_label=None, checkpoint_meta=None):
    """Train REINFORCE with learned baseline (actor-critic)."""
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    actor = GaussianPolicy(env.obs_dim, env.act_dim, hidden).to(device)
    critic = ValueNetwork(env.obs_dim, hidden).to(device)
    actor_opt = optim.Adam(actor.parameters(), lr=lr_actor)
    critic_opt = optim.Adam(critic.parameters(), lr=lr_critic)
    rewards_history = []
    adv_std_history = []

    t0 = time.time()
    for ep in range(n_episodes):
        obs = env.reset()
        log_probs = []
        rewards = []
        states = []

        while True:
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            action, log_prob = actor.get_action(obs_t)
            act_np = action.squeeze(0).cpu().detach().numpy()

            next_obs, reward, terminated, truncated, _ = env.step(act_np)
            log_probs.append(log_prob)
            rewards.append(reward)
            states.append(obs)
            obs = next_obs
            if terminated or truncated:
                break

        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)

        states_t = torch.FloatTensor(np.array(states)).to(device)
        values = critic(states_t)

        advantages = returns - values.detach()
        adv_std_history.append(float(advantages.std().item()) if len(advantages) > 1 else 0.0)
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs_t = torch.stack(log_probs).squeeze()
        actor_loss = -(log_probs_t * advantages).mean()

        actor_opt.zero_grad()
        actor_loss.backward()
        actor_opt.step()

        critic_loss = nn.functional.mse_loss(values, returns)

        critic_opt.zero_grad()
        critic_loss.backward()
        critic_opt.step()

        total_reward = sum(rewards)
        rewards_history.append(total_reward)

        if verbose and (ep + 1) % 100 == 0:
            avg = np.mean(rewards_history[-50:])
            std_val = actor.log_std.exp().mean().item()
            print(f'  Episode {ep+1:4d} | Avg reward: {avg:7.1f} | std: {std_val:.3f}')

    elapsed = time.time() - t0
    if verbose:
        print(f'  Training completed in {elapsed:.1f}s')

    if checkpoint_label is not None:
        meta = {
            'obs_dim': int(env.obs_dim),
            'act_dim': int(env.act_dim),
            'hidden': int(hidden),
            'critic_hidden': int(hidden),
        }
        if checkpoint_meta:
            meta.update(checkpoint_meta)
        save_pg_checkpoint({
            'kind': 'actor_critic',
            'label': checkpoint_label,
            'meta': meta,
            'policy_state_dict': actor.state_dict(),
            'critic_state_dict': critic.state_dict(),
            'rewards': np.asarray(rewards_history, dtype=np.float32),
            'extras': {
                'adv_std_history': np.asarray(adv_std_history, dtype=np.float32),
            },
        })

    return actor, critic, rewards_history, adv_std_history


## Reuse the Demo 6 / Demo 7 results

We keep the same seed list and same episode count as Demo 7.
If the old checkpoints are present, the notebook loads them directly.

In [ ]:
# ---------- Demo 6: Baseline comparison averaged over 10 seeds ----------
def discounted_returns(rewards, gamma=0.99):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return np.array(returns, dtype=np.float32)


def train_reinforce_with_baseline(env, baseline='none', n_episodes=500, gamma=0.99,
                                  lr=1e-3, hidden=64, verbose=True, seed=None,
                                  checkpoint_label=None, checkpoint_meta=None):
    """Train REINFORCE with a hand-designed baseline. Returns (policy, rewards, adv_std)."""
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    policy = GaussianPolicy(env.obs_dim, env.act_dim, hidden).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    rewards_history = []
    adv_std_history = []
    running_return_mean = 0.0
    time_baseline_sum = []
    time_baseline_count = []

    t0 = time.time()
    for ep in range(n_episodes):
        obs = env.reset()
        log_probs = []
        rewards = []

        while True:
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            action, log_prob = policy.get_action(obs_t)
            act_np = action.squeeze(0).cpu().detach().numpy()

            next_obs, reward, terminated, truncated, _ = env.step(act_np)
            log_probs.append(log_prob)
            rewards.append(reward)
            obs = next_obs
            if terminated or truncated:
                break

        returns_np = discounted_returns(rewards, gamma)
        baseline_np = np.zeros_like(returns_np)

        if baseline == 'constant' and ep > 0:
            baseline_np[:] = running_return_mean
        elif baseline == 'time':
            for t in range(len(returns_np)):
                if t < len(time_baseline_sum) and time_baseline_count[t] > 0:
                    baseline_np[t] = time_baseline_sum[t] / time_baseline_count[t]

        advantages_np = returns_np - baseline_np
        adv_std_history.append(float(np.std(advantages_np)) if len(advantages_np) > 1 else 0.0)

        advantages = torch.FloatTensor(advantages_np).to(device)
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs_t = torch.stack(log_probs).squeeze()
        loss = -(log_probs_t * advantages).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_reward = sum(rewards)
        rewards_history.append(total_reward)

        if baseline == 'constant':
            episode_return = float(returns_np[0])
            running_return_mean = (running_return_mean * ep + episode_return) / (ep + 1)
        elif baseline == 'time':
            while len(time_baseline_sum) < len(returns_np):
                time_baseline_sum.append(0.0)
                time_baseline_count.append(0)
            for t, G in enumerate(returns_np):
                time_baseline_sum[t] += float(G)
                time_baseline_count[t] += 1

        if verbose and (ep + 1) % 100 == 0:
            avg_r = np.mean(rewards_history[-50:])
            avg_std = np.mean(adv_std_history[-50:])
            print(f'  Episode {ep+1:4d} | Avg reward: {avg_r:7.1f} | avg std(G-b): {avg_std:7.2f}')

    elapsed = time.time() - t0
    if verbose:
        print(f'  Training completed in {elapsed:.1f}s')

    if checkpoint_label is not None:
        meta = {
            'obs_dim': int(env.obs_dim),
            'act_dim': int(env.act_dim),
            'hidden': int(hidden),
            'baseline': baseline,
        }
        if checkpoint_meta:
            meta.update(checkpoint_meta)
        save_pg_checkpoint({
            'kind': 'reinforce_baseline',
            'label': checkpoint_label,
            'meta': meta,
            'policy_state_dict': policy.state_dict(),
            'rewards': np.asarray(rewards_history, dtype=np.float32),
            'extras': {
                'adv_std_history': np.asarray(adv_std_history, dtype=np.float32),
            },
        })

    return policy, rewards_history, adv_std_history


print('=== Demo 6: Baseline choices averaged over 10 seeds ===\n')

compare_seeds = list(range(10))
compare_episodes = 600
baseline_specs = [
    ('Vanilla (no baseline)', 'none', 'tab:orange'),
    ('Constant baseline', 'constant', 'tab:green'),
    ('Time-dependent baseline', 'time', 'tab:purple'),
]

baseline_results = {}
for label, mode, color in baseline_specs:
    print(f'--- {label} ---')
    policies_by_seed = {}
    reward_histories_by_seed = {}
    adv_std_histories_by_seed = {}

    for seed in compare_seeds:
        env_cmp = CrawlerEnv()
        baseline_meta = {
            'demo': '6',
            'variant': 'baseline_compare',
            'baseline': mode,
            'seed': seed,
            'n_episodes': compare_episodes,
        }
        latest = maybe_load_pg_checkpoint(
            kind='reinforce_baseline',
            obs_dim=env_cmp.obs_dim,
            act_dim=env_cmp.act_dim,
            hidden=64,
            label=f'{label} (seed {seed})',
            extra_meta=baseline_meta,
        )
        if latest is None:
            print(f'  Seed {seed}: training')
            policy_cmp, rewards_cmp, adv_std_cmp = train_reinforce_with_baseline(
                env_cmp,
                baseline=mode,
                n_episodes=compare_episodes,
                lr=1e-3,
                seed=seed,
                verbose=False,
                checkpoint_label=f'Demo 6: {label} (seed {seed})',
                checkpoint_meta=baseline_meta,
            )
        else:
            print(f'  Seed {seed}: loaded')
            policy_cmp, rewards_cmp, extras = latest
            adv_std_cmp = list(extras.get('adv_std_history', []))

        policies_by_seed[seed] = policy_cmp
        reward_histories_by_seed[seed] = rewards_cmp
        adv_std_histories_by_seed[seed] = adv_std_cmp

    baseline_results[label] = {
        'policies_by_seed': policies_by_seed,
        'reward_histories_by_seed': reward_histories_by_seed,
        'adv_std_histories_by_seed': adv_std_histories_by_seed,
        'color': color,
    }
    print()


In [ ]:
# ---------- Demo 7: Actor-Critic on the crawler ----------
print('=== Demo 7: Learned state-dependent baseline (Actor-Critic, 10 seeds) ===')
print(f'Actor: Gaussian policy (same architecture as Demo 5)')
print(f'Critic: value network V(s) as baseline')
print(f'Advantage: G_t - V(s_t) -- reduces variance!\n')

ac_seeds = compare_seeds
ac_episodes = compare_episodes
actor_critic_results = {
    'policies_by_seed': {},
    'critics_by_seed': {},
    'reward_histories_by_seed': {},
    'adv_std_histories_by_seed': {},
}

for seed in ac_seeds:
    env_ac = CrawlerEnv()
    actor_critic_meta = {
        'demo': '7',
        'variant': 'actor_critic',
        'seed': seed,
        'n_episodes': ac_episodes,
    }
    latest = maybe_load_pg_checkpoint(
        kind='actor_critic',
        obs_dim=env_ac.obs_dim,
        act_dim=env_ac.act_dim,
        hidden=64,
        label=f'Demo 7: Actor-Critic (seed {seed})',
        extra_meta=actor_critic_meta,
        with_critic=True,
    )
    if latest is None:
        print(f'  Seed {seed}: training')
        actor_seed, critic_seed, rewards_seed, adv_std_seed = train_actor_critic(
            env_ac,
            n_episodes=ac_episodes,
            lr_actor=1e-3,
            seed=seed,
            verbose=False,
            checkpoint_label=f'Demo 7: Actor-Critic (seed {seed})',
            checkpoint_meta=actor_critic_meta,
        )
    else:
        print(f'  Seed {seed}: loaded')
        actor_seed, critic_seed, rewards_seed, extras = latest
        adv_std_seed = list(extras.get('adv_std_history', []))

    actor_critic_results['policies_by_seed'][seed] = actor_seed
    actor_critic_results['critics_by_seed'][seed] = critic_seed
    actor_critic_results['reward_histories_by_seed'][seed] = rewards_seed
    actor_critic_results['adv_std_histories_by_seed'][seed] = adv_std_seed

actor_ac_seed = max(
    ac_seeds,
    key=lambda s: np.mean(actor_critic_results['reward_histories_by_seed'][s][-50:])
)
actor_ac = actor_critic_results['policies_by_seed'][actor_ac_seed]
critic_ac = actor_critic_results['critics_by_seed'][actor_ac_seed]
rewards_ac = actor_critic_results['reward_histories_by_seed'][actor_ac_seed]
adv_std_ac = actor_critic_results['adv_std_histories_by_seed'][actor_ac_seed]
print(f'Using seed {actor_ac_seed} as the representative Actor-Critic policy for rollout cells.')


## PPO: Standalone Algorithm Focus

Before comparing PPO against the older policy-gradient baselines, it helps to isolate the algorithm itself.
The goal of this section is to answer three questions clearly:

1. What are the **state, action, and reward** in this crawler setup?
2. What extra ingredients does PPO add on top of Actor-Critic?
3. How does the **clipped PPO update** map into code?

**Setup at a glance**

| Quantity | Notation | In this PPO demo |
|---|---|---|
| State | $s_t$ | 4D continuous crawler observation: arm angle, hand angle, arm angular velocity, hand angular velocity |
| Action | $a_t$ | 2D continuous motor torques, one for each joint, clipped to $[-1, 1]$ |
| Reward | $r_t$ | Forward torso velocity: $$r_t = \frac{x_{t+1} - x_t}{\Delta t}$$ |
| Policy | $\pi_\theta(a\mid s)$ | Gaussian actor with neural-network mean and learned log-standard-deviation |
| Value baseline | $V_\phi(s)$ | Critic network trained to predict return |
| PPO-specific idea | $\rho_t(\theta)$ | Probability ratio is clipped so one update cannot move the policy too far |

**What PPO changes relative to Demo 7**

- Demo 7 already used a learned value baseline, so PPO is **not** starting from vanilla REINFORCE.
- The new ingredient is the **clipped policy-ratio objective**: if the new policy wants to change action probabilities too aggressively, PPO truncates the update.
- This usually lowers update variance and reduces destructive jumps, at the cost of sometimes being more conservative.

**Pseudocode**

```text
initialize actor parameters θ and critic parameters ϕ
for episode = 1, 2, ..., N:
    run the current stochastic policy π_θ on the crawler
    collect states s_t, actions a_t, rewards r_t, old log-probs log π_θ_old(a_t | s_t), values V_ϕ(s_t)

    compute GAE advantages Â_t and return targets R_t
    normalize the advantages within this batch

    for several PPO epochs:
        evaluate the current policy on the stored states/actions
        compute ratio ρ_t = π_θ(a_t | s_t) / π_θ_old(a_t | s_t)
        compute clipped actor objective:
            L_actor = - mean( min(ρ_t Â_t, clip(ρ_t, 1-ε, 1+ε) Â_t) )
        compute critic regression loss:
            L_critic = mean( V_ϕ(s_t) - R_t )^2
        update θ and ϕ with gradient descent
```

Recommended tuning order for this notebook:

1. Fix the PPO action-probability bookkeeping so the stored `old_log_probs` match the actions actually sent to the environment.
2. Increase the rollout batch size so each PPO update sees several episodes, not just one trajectory.
3. If learning is still too slow, relax the clipping / update conservativeness.
4. Only then sweep actor / critic learning rates and initial exploration scale.

The code cells below implement those first two changes and put a runnable quick-sweep right after the standalone PPO block.

In [ ]:
# ============================================================
# PPO (clipped objective + value baseline)
# ============================================================

def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    rewards = np.asarray(rewards, dtype=np.float32)
    values = np.asarray(values, dtype=np.float32)
    advantages = np.zeros_like(rewards, dtype=np.float32)
    gae = 0.0
    next_value = 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae
        next_value = values[t]
    returns = advantages + values
    return advantages, returns


def train_ppo(env, n_episodes=600, gamma=0.99, lam=0.95,
              lr_actor=3e-4, lr_critic=1e-3, hidden=64,
              clip_eps=0.2, train_epochs=4, minibatch_size=500,
              batch_episodes=8, value_coef=0.5,
              max_grad_norm=0.5, init_log_std=-1.0,
              verbose=True, seed=None, checkpoint_label=None, checkpoint_meta=None):
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    actor = GaussianPolicy(env.obs_dim, env.act_dim, hidden, init_log_std=init_log_std).to(device)
    critic = ValueNetwork(env.obs_dim, hidden).to(device)
    actor_opt = optim.Adam(actor.parameters(), lr=lr_actor)
    critic_opt = optim.Adam(critic.parameters(), lr=lr_critic)

    rewards_history = []
    adv_std_history = []

    run_label = checkpoint_label or 'PPO'
    t0 = time.time()
    episodes_seen = 0
    while episodes_seen < n_episodes:
        batch_states = []
        batch_actions = []
        batch_old_log_probs = []
        batch_advantages = []
        batch_returns = []
        episodes_this_batch = min(batch_episodes, n_episodes - episodes_seen)

        for _ in range(episodes_this_batch):
            obs = env.reset()
            states = []
            actions = []
            rewards = []
            old_log_probs = []
            values = []

            while True:
                obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                with torch.no_grad():
                    action_t, log_prob_t = actor.get_action(obs_t)
                    value_t = critic(obs_t)
                action_np = action_t.squeeze(0).cpu().numpy()
                next_obs, reward, terminated, truncated, _ = env.step(action_np)

                states.append(obs)
                actions.append(action_np)
                rewards.append(reward)
                old_log_probs.append(float(log_prob_t.item()))
                values.append(float(value_t.item()))

                obs = next_obs
                if terminated or truncated:
                    break

            advantages_np, returns_np = compute_gae(rewards, values, gamma=gamma, lam=lam)
            adv_std_history.append(float(np.std(advantages_np)) if len(advantages_np) > 1 else 0.0)
            rewards_history.append(float(np.sum(rewards)))

            batch_states.extend(states)
            batch_actions.extend(actions)
            batch_old_log_probs.extend(old_log_probs)
            batch_advantages.extend(advantages_np.tolist())
            batch_returns.extend(returns_np.tolist())
            episodes_seen += 1

        advantages_t = torch.FloatTensor(np.asarray(batch_advantages, dtype=np.float32)).to(device)
        if len(advantages_t) > 1:
            advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        returns_t = torch.FloatTensor(np.asarray(batch_returns, dtype=np.float32)).to(device)
        states_t = torch.FloatTensor(np.asarray(batch_states, dtype=np.float32)).to(device)
        actions_t = torch.FloatTensor(np.asarray(batch_actions, dtype=np.float32)).to(device)
        old_log_probs_t = torch.FloatTensor(np.asarray(batch_old_log_probs, dtype=np.float32)).to(device)

        n = len(batch_states)
        idx = np.arange(n)
        effective_minibatch = min(minibatch_size, n)
        for _ in range(train_epochs):
            np.random.shuffle(idx)
            for start in range(0, n, effective_minibatch):
                mb_idx = idx[start:start + effective_minibatch]
                mb_states = states_t[mb_idx]
                mb_actions = actions_t[mb_idx]
                mb_old_log_probs = old_log_probs_t[mb_idx]
                mb_advantages = advantages_t[mb_idx]
                mb_returns = returns_t[mb_idx]

                new_log_probs = actor.evaluate(mb_states, mb_actions)
                ratio = torch.exp(new_log_probs - mb_old_log_probs)
                clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)
                actor_loss = -torch.min(ratio * mb_advantages, clipped_ratio * mb_advantages).mean()

                actor_opt.zero_grad()
                actor_loss.backward()
                nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
                actor_opt.step()

                value_pred = critic(mb_states)
                critic_loss = value_coef * nn.functional.mse_loss(value_pred, mb_returns)
                critic_opt.zero_grad()
                critic_loss.backward()
                nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
                critic_opt.step()

        if verbose and episodes_seen % 100 == 0:
            avg_r = np.mean(rewards_history[-50:])
            avg_adv_std = np.mean(adv_std_history[-50:])
            print(f'  {run_label} | Episode {episodes_seen:4d} | Avg reward: {avg_r:7.1f} | avg std(A): {avg_adv_std:7.2f}')

    elapsed = time.time() - t0
    if verbose:
        print(f'  Training completed in {elapsed:.1f}s')

    if checkpoint_label is not None:
        meta = {
            'obs_dim': int(env.obs_dim),
            'act_dim': int(env.act_dim),
            'hidden': int(hidden),
            'critic_hidden': int(hidden),
            'batch_episodes': int(batch_episodes),
            'clip_eps': float(clip_eps),
            'train_epochs': int(train_epochs),
            'lr_actor': float(lr_actor),
            'lr_critic': float(lr_critic),
            'init_log_std': float(init_log_std),
        }
        if checkpoint_meta:
            meta.update(checkpoint_meta)
        save_pg_checkpoint({
            'kind': 'ppo',
            'label': checkpoint_label,
            'meta': meta,
            'policy_state_dict': actor.state_dict(),
            'critic_state_dict': critic.state_dict(),
            'rewards': np.asarray(rewards_history, dtype=np.float32),
            'extras': {
                'adv_std_history': np.asarray(adv_std_history, dtype=np.float32),
            },
        })

    return actor, critic, rewards_history, adv_std_history


## PPO Tuning Runner

Run this quick sweep immediately after the standalone PPO definition if you want to test the tuning knobs before doing the full lecture comparison.

Suggested workflow:
- Start with 3 to 4 seeds and compare all presets on the same plot.
- If one configuration is clearly better, promote only that winner to a wider confirmation run.
- Treat the selected PPO config below as the representative one for the later comparison cells.


In [ ]:
# ============================================================
# PPO benchmark on the shared crawler protocol
# ============================================================

PPO_TUNING_CONFIG = {
    'n_episodes': compare_episodes,
    'batch_episodes': 8,
    'clip_eps': 0.2,
    'train_epochs': 4,
    'minibatch_size': 500,
    'lr_actor': 3e-4,
    'lr_critic': 1e-3,
    'value_coef': 0.5,
    'max_grad_norm': 0.5,
    'init_log_std': -1.0,
}

PPO_SWEEP_PRESETS = [
    {
        'name': 'batch8_clip0.2_epochs4_lr3e-4_1e-3_std-1.0',
        **PPO_TUNING_CONFIG,
    },
    {
        'name': 'batch8_clip0.25_epochs4_lr3e-4_1e-3_std-1.0',
        **PPO_TUNING_CONFIG,
        'clip_eps': 0.25,
    },
    {
        'name': 'batch8_clip0.2_epochs8_lr3e-4_1e-3_std-1.0',
        **PPO_TUNING_CONFIG,
        'train_epochs': 8,
    },
    {
        'name': 'batch16_clip0.2_epochs4_lr1e-4_3e-4_std-1.0',
        **PPO_TUNING_CONFIG,
        'batch_episodes': 16,
        'lr_actor': 1e-4,
        'lr_critic': 3e-4,
    },
]

print(f"=== L7-1: PPO on the same crawler ({len(range(4))} quick-sweep seeds) ===")
print('Actor: Gaussian policy')
print('Critic: learned value baseline V(s)')
print('Extra idea: clipped policy-ratio update for more stable policy improvement')
ppo_seeds = list(range(4))
ppo_sweep_results = {}
ppo_curve_colors = ['tab:red', 'tab:orange', 'tab:green', 'tab:purple', 'tab:brown', 'tab:pink']

for ppo_config in PPO_SWEEP_PRESETS:
    ppo_config = ppo_config.copy()
    ppo_episodes = ppo_config['n_episodes']
    print(f"\n=== PPO config: {ppo_config['name']} ===")
    config_results = {
        'config': ppo_config.copy(),
        'policies_by_seed': {},
        'critics_by_seed': {},
        'reward_histories_by_seed': {},
        'adv_std_histories_by_seed': {},
    }

    for seed in ppo_seeds:
        env_ppo = CrawlerEnv()
        ppo_meta = {
            'demo': 'L7-1',
            'variant': 'ppo',
            'seed': seed,
            'n_episodes': ppo_episodes,
            'batch_episodes': ppo_config['batch_episodes'],
            'clip_eps': ppo_config['clip_eps'],
            'train_epochs': ppo_config['train_epochs'],
            'lr_actor': ppo_config['lr_actor'],
            'lr_critic': ppo_config['lr_critic'],
            'init_log_std': ppo_config['init_log_std'],
            'config_name': ppo_config['name'],
        }
        ppo_checkpoint_label = f"L7-1: PPO [{ppo_config['name']}] (seed {seed})"
        latest = maybe_load_pg_checkpoint(
            kind='ppo',
            obs_dim=env_ppo.obs_dim,
            act_dim=env_ppo.act_dim,
            hidden=64,
            label=ppo_checkpoint_label,
            extra_meta=ppo_meta,
            with_critic=True,
        )
        if latest is None:
            print(f'  Seed {seed}: training')
            actor_seed, critic_seed, rewards_seed, adv_std_seed = train_ppo(
                env_ppo,
                n_episodes=ppo_episodes,
                batch_episodes=ppo_config['batch_episodes'],
                clip_eps=ppo_config['clip_eps'],
                train_epochs=ppo_config['train_epochs'],
                minibatch_size=ppo_config['minibatch_size'],
                lr_actor=ppo_config['lr_actor'],
                lr_critic=ppo_config['lr_critic'],
                value_coef=ppo_config['value_coef'],
                max_grad_norm=ppo_config['max_grad_norm'],
                init_log_std=ppo_config['init_log_std'],
                seed=seed,
                verbose=True,
                checkpoint_label=ppo_checkpoint_label,
                checkpoint_meta=ppo_meta,
            )
        else:
            print(f'  Seed {seed}: loaded')
            actor_seed, critic_seed, rewards_seed, extras = latest
            adv_std_seed = list(extras.get('adv_std_history', []))

        config_results['policies_by_seed'][seed] = actor_seed
        config_results['critics_by_seed'][seed] = critic_seed
        config_results['reward_histories_by_seed'][seed] = rewards_seed
        config_results['adv_std_histories_by_seed'][seed] = adv_std_seed

    ppo_sweep_results[ppo_config['name']] = config_results

fig, ax = plt.subplots(figsize=(11, 5))
for idx, (config_name, config_results) in enumerate(ppo_sweep_results.items()):
    plot_seed_average(
        ax,
        list(config_results['reward_histories_by_seed'].values()),
        color=ppo_curve_colors[idx % len(ppo_curve_colors)],
        label=config_name,
        window=40,
        alpha=0.12,
    )
ax.set_title('PPO quick sweep: reward curves by configuration', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Final PPO config comparison across quick-sweep seeds (last 50 episodes):')
for config_name, config_results in ppo_sweep_results.items():
    reward_mean, reward_std = summarize_final_window(
        list(config_results['reward_histories_by_seed'].values()),
        tail=50,
    )
    print(f'{config_name:45s} | reward: {reward_mean:6.1f} +/- {reward_std:4.1f}')

best_ppo_config_name = max(
    ppo_sweep_results,
    key=lambda name: summarize_final_window(
        list(ppo_sweep_results[name]['reward_histories_by_seed'].values()),
        tail=50,
    )[0],
)
ppo_results = ppo_sweep_results[best_ppo_config_name]
ppo_config = ppo_results['config'].copy()
print(f"\nSelected PPO config for downstream cells: {best_ppo_config_name}")

ppo_seed = max(
    ppo_results['reward_histories_by_seed'],
    key=lambda s: np.mean(ppo_results['reward_histories_by_seed'][s][-50:])
)
ppo_actor = ppo_results['policies_by_seed'][ppo_seed]
ppo_critic = ppo_results['critics_by_seed'][ppo_seed]
rewards_ppo = ppo_results['reward_histories_by_seed'][ppo_seed]
adv_std_ppo = ppo_results['adv_std_histories_by_seed'][ppo_seed]
print(f'Using seed {ppo_seed} as the representative PPO policy for rollout cells.')


In [ ]:
# PPO vs previous baselines on the same crawler protocol
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
window = 40

ax = axes[0]
for label, result in baseline_results.items():
    plot_seed_average(
        ax,
        list(result['reward_histories_by_seed'].values()),
        color=result['color'],
        label=label,
        window=window,
    )
plot_seed_average(
    ax,
    list(actor_critic_results['reward_histories_by_seed'].values()),
    color='tab:blue',
    label='Actor-Critic (learned $V(s)$)',
    window=window,
)
plot_seed_average(
    ax,
    list(ppo_results['reward_histories_by_seed'].values()),
    color='tab:red',
    label='PPO (clipped objective)',
    window=window,
)
ax.set_title('Reward curves: mean +/- 1 std over 10 seeds', fontsize=13, fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

ax = axes[1]
for label, result in baseline_results.items():
    plot_seed_average(
        ax,
        list(result['adv_std_histories_by_seed'].values()),
        color=result['color'],
        label=label,
        window=window,
    )
plot_seed_average(
    ax,
    list(actor_critic_results['adv_std_histories_by_seed'].values()),
    color='tab:blue',
    label='Actor-Critic (learned $V(s)$)',
    window=window,
)
plot_seed_average(
    ax,
    list(ppo_results['adv_std_histories_by_seed'].values()),
    color='tab:red',
    label='PPO (raw GAE advantage std)',
    window=window,
)
ax.set_xlabel('Episode')
ax.set_ylabel('Std of raw advantage weights')
ax.set_title('Variance proxy: mean +/- 1 std over 10 seeds', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Final comparison across 10 seeds (last 50 episodes):')
for label, result in baseline_results.items():
    reward_mean, reward_std = summarize_final_window(list(result['reward_histories_by_seed'].values()), tail=50)
    var_mean, var_std = summarize_final_window(list(result['adv_std_histories_by_seed'].values()), tail=50)
    print(f'{label:24s} | reward: {reward_mean:6.1f} +/- {reward_std:4.1f} | std: {var_mean:6.2f} +/- {var_std:4.2f}')

ac_reward_mean, ac_reward_std = summarize_final_window(list(actor_critic_results['reward_histories_by_seed'].values()), tail=50)
ac_var_mean, ac_var_std = summarize_final_window(list(actor_critic_results['adv_std_histories_by_seed'].values()), tail=50)
print(f"{'Actor-Critic (learned)':24s} | reward: {ac_reward_mean:6.1f} +/- {ac_reward_std:4.1f} | std: {ac_var_mean:6.2f} +/- {ac_var_std:4.2f}")

ppo_reward_mean, ppo_reward_std = summarize_final_window(list(ppo_results['reward_histories_by_seed'].values()), tail=50)
ppo_var_mean, ppo_var_std = summarize_final_window(list(ppo_results['adv_std_histories_by_seed'].values()), tail=50)
print(f"{'PPO (clipped objective)':24s} | reward: {ppo_reward_mean:6.1f} +/- {ppo_reward_std:4.1f} | std: {ppo_var_mean:6.2f} +/- {ppo_var_std:4.2f}")


In [ ]:
# 10-second rollout of the representative PPO policy
env_eval_ppo = CrawlerEnv()

def ppo_policy(obs):
    with torch.no_grad():
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        mu, _ = ppo_actor(obs_t)
        return mu.squeeze(0).cpu().numpy()

frames_ppo, dist_ppo, _ = eval_policy(env_eval_ppo, ppo_policy, 'L7-1: PPO')
show_video(frames_ppo, title=f'PPO — {dist_ppo:.2f}m in 10s')


In [ ]:
# Final comparison: representative policies from the policy-gradient family
policy_panels = []
comparison_results = {}

def add_policy_panel(label, policy_fn, *, env=None, video_title=None, max_steps=500,
                     frames=None, dist=None):
    if frames is None or dist is None:
        if env is None:
            env = CrawlerEnv(max_steps=max_steps)
        frames, dist, _ = eval_policy(env, policy_fn, label, max_steps=max_steps)
    comparison_results[label] = dist
    video_html = show_video(frames, title=video_title or f'{label} — {dist:.2f}m in 10s').data
    policy_panels.append({
        'label': label,
        'distance': dist,
        'video_html': video_html,
    })

def representative_seed(history_dict):
    return max(history_dict, key=lambda s: np.mean(history_dict[s][-50:]))

def make_comparison_plot_base64(results_dict, title='Policy Comparison: Distance Traveled in 10s'):
    labels = list(results_dict.keys())
    dists = list(results_dict.values())
    colors = ['#d9534f' if d < 0.5 else '#5bc0de' if d < 1.5 else '#5cb85c' for d in dists]
    fig, ax = plt.subplots(figsize=(9, max(4, len(labels) * 0.55 + 1.2)))
    bars = ax.barh(labels, dists, color=colors, edgecolor='white', height=0.6)
    ax.set_xlabel('Distance (m)')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='gray', linewidth=0.5)
    for bar, d in zip(bars, dists):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{d:.2f}m', va='center', fontsize=11)
    ax.invert_yaxis()
    plt.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=160, bbox_inches='tight')
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode('ascii')

reinforce_meta = {
    'demo': '5',
    'variant': 'reinforce',
    'n_episodes': 4500,
}
env_tmp = CrawlerEnv()
loaded = maybe_load_pg_checkpoint(
    kind='reinforce',
    obs_dim=env_tmp.obs_dim,
    act_dim=env_tmp.act_dim,
    hidden=64,
    label='Demo 5: REINFORCE',
    extra_meta=reinforce_meta,
)
if loaded is None:
    raise RuntimeError('Expected the Demo 5 REINFORCE checkpoint to exist, but none was found.')
policy_reinforce, rewards_reinforce, reinforce_extras = loaded

def reinforce_policy(obs):
    with torch.no_grad():
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        mu, _ = policy_reinforce(obs_t)
        return mu.squeeze(0).cpu().numpy()

const_seed = representative_seed(baseline_results['Constant baseline']['reward_histories_by_seed'])
const_policy_model = baseline_results['Constant baseline']['policies_by_seed'][const_seed]
def constant_baseline_policy(obs):
    with torch.no_grad():
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        mu, _ = const_policy_model(obs_t)
        return mu.squeeze(0).cpu().numpy()

time_seed = representative_seed(baseline_results['Time-dependent baseline']['reward_histories_by_seed'])
time_policy_model = baseline_results['Time-dependent baseline']['policies_by_seed'][time_seed]
def time_baseline_policy(obs):
    with torch.no_grad():
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        mu, _ = time_policy_model(obs_t)
        return mu.squeeze(0).cpu().numpy()

def actor_critic_policy(obs):
    with torch.no_grad():
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        mu, _ = actor_ac(obs_t)
        return mu.squeeze(0).cpu().numpy()

if 'frames_ppo' in globals() and 'dist_ppo' in globals():
    add_policy_panel('L7-1: PPO', ppo_policy, video_title='PPO', frames=frames_ppo, dist=dist_ppo)
else:
    add_policy_panel('L7-1: PPO', ppo_policy, video_title='PPO')
add_policy_panel('Demo 5: REINFORCE', reinforce_policy, video_title='REINFORCE')
add_policy_panel('Demo 6: Constant baseline', constant_baseline_policy, video_title='Constant baseline')
add_policy_panel('Demo 6: Time baseline', time_baseline_policy, video_title='Time-dependent baseline')
add_policy_panel('Demo 7: Actor-Critic', actor_critic_policy, video_title='Actor-Critic')

plot_png = make_comparison_plot_base64(comparison_results)
cards_html = ''.join(
    f"<div style='background:#fff;border:1px solid #ddd;border-radius:10px;padding:10px;'>"
    f"<div style='font-weight:600;margin-bottom:8px'>{panel['label']} ({panel['distance']:.2f}m)</div>"
    f"{panel['video_html']}"
    f"</div>"
    for panel in policy_panels
)
display(HTML(
    f"""
<div style='display:grid;grid-template-columns:minmax(340px, 420px) 1fr;gap:20px;align-items:start;'>
  <div style='background:#fff;border:1px solid #ddd;border-radius:10px;padding:12px;'>
    <div style='font-weight:700;font-size:18px;margin-bottom:10px;'>Distance traveled in 10 seconds</div>
    <img src='data:image/png;base64,{plot_png}' style='width:100%;height:auto;display:block;' />
  </div>
  <div style='display:flex;flex-direction:column;gap:16px;'>
    {cards_html}
  </div>
</div>
"""
))


## Scaling Test: Humanoid 3D

The crawler only has **2 actions** and short, low-dimensional trajectories. To test whether PPO really scales better, we need a harder control problem.

`Humanoid-v5` is a much stronger stress test:
- The action space is **17-dimensional** instead of 2-dimensional.
- The observation is much larger, and the body has many more coupled joints.
- Episodes are longer, so delayed credit assignment matters more.

The comparison below uses **environment steps** on the x-axis and then evaluates the trained policies on a shared rollout seed.
This time we include two REINFORCE-style baselines: a **learned value baseline** and a simpler **time-dependent baseline**.
The intended lesson is practical rather than theoretical: on this higher-dimensional task, the simpler REINFORCE baseline can break down badly, the learned baseline does better but still plateaus sooner, and PPO can keep improving when we let it train longer.
So the Humanoid section is set up as a **baseline quality + continued-improvement** comparison rather than a strict equal-budget benchmark.

In [65]:
# ============================================================
# Humanoid 3D helpers
# ============================================================

HUMANOID_ENV_ID = 'Humanoid-v5'
HUMANOID_SEED = 0
HUMANOID_BASELINE_STEP_BUDGET = 100_000
HUMANOID_PPO_STEP_BUDGET = 300_000


class ScaledGaussianPolicy(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128, init_log_std=-1.0, action_scale=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim), nn.Tanh(),
        )
        self.log_std = nn.Parameter(torch.full((act_dim,), float(init_log_std)))
        self.action_scale = float(action_scale)

    def forward(self, obs):
        mu = self.net(obs) * self.action_scale
        std = self.log_std.exp() * self.action_scale
        return mu, std

    def get_action(self, obs):
        mu, std = self.forward(obs)
        dist = Normal(mu, std)
        raw_action = dist.sample()
        log_prob = dist.log_prob(raw_action).sum(dim=-1)
        return raw_action.clamp(-self.action_scale, self.action_scale), log_prob


class EpisodeStatsCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []
        self.timesteps = []
        self.episode_distances = []

    def _on_step(self):
        infos = self.locals.get('infos', [])
        dones = self.locals.get('dones', [])
        for done, info in zip(dones, infos):
            if done and 'episode' in info:
                self.episode_rewards.append(float(info['episode']['r']))
                self.episode_lengths.append(int(info['episode']['l']))
                self.timesteps.append(int(self.num_timesteps))
                self.episode_distances.append(float(info.get('x_position', np.nan)))
        return True


def make_humanoid_env(*, seed=None, render_mode=None):
    env = gym.make(HUMANOID_ENV_ID, render_mode=render_mode)
    if seed is not None:
        env.reset(seed=seed)
    return env


def make_humanoid_vec_env(*, n_envs=4, seed=0):
    def make_single(rank):
        def thunk():
            env = gym.make(HUMANOID_ENV_ID)
            env.reset(seed=seed + rank)
            return env
        return thunk

    vec_env = DummyVecEnv([make_single(rank) for rank in range(n_envs)])
    vec_env = VecMonitor(vec_env)
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0, gamma=0.99)
    return vec_env


def humanoid_pg_meta(variant, *, seed, step_budget):
    return {
        'demo': 'L7-1',
        'variant': variant,
        'config_name': f'{HUMANOID_ENV_ID}_steps{step_budget}',
        'seed': int(seed),
        'step_budget': int(step_budget),
    }


def humanoid_ppo_meta(*, seed, total_timesteps, n_envs):
    return {
        'demo': 'L7-1',
        'variant': 'humanoid_ppo',
        'config_name': f'{HUMANOID_ENV_ID}_n_envs{n_envs}_steps{total_timesteps}_distance_tracking_v2',
        'seed': int(seed),
        'step_budget': int(total_timesteps),
        'n_envs': int(n_envs),
    }


def maybe_load_humanoid_pg_checkpoint(*, kind, obs_dim, act_dim, hidden, action_scale, label,
                                      extra_meta=None, with_critic=True):
    if FORCE_RETRAIN:
        print(f'FORCE_RETRAIN=True for {label}; ignoring saved checkpoint.')
        return None

    required_meta = {
        'obs_dim': int(obs_dim),
        'act_dim': int(act_dim),
        'hidden': int(hidden),
        'critic_hidden': int(hidden),
        'action_scale': float(action_scale),
    }
    if extra_meta:
        required_meta.update(extra_meta)

    path = checkpoint_path_for(kind, required_meta)
    payload = load_pg_checkpoint(path, verbose=False)
    if not checkpoint_matches(payload, kind=kind, required_meta=required_meta):
        return None

    actor = ScaledGaussianPolicy(
        obs_dim,
        act_dim,
        hidden,
        init_log_std=float(required_meta.get('init_log_std', -1.0)),
        action_scale=action_scale,
    ).to(device)
    actor.load_state_dict(payload['policy_state_dict'])
    actor.eval()

    rewards = list(payload.get('rewards', []))
    extras = payload.get('extras', {})
    print(f'Loaded checkpoint for {label}; skipping retraining.')
    print(f'  path: {path}')

    if with_critic:
        critic = ValueNetwork(obs_dim, hidden).to(device)
        critic.load_state_dict(payload['critic_state_dict'])
        critic.eval()
        return actor, critic, rewards, extras

    return actor, rewards, extras


def humanoid_ppo_paths(meta):
    slug = checkpoint_slug('humanoid_ppo', meta)
    return {
        'model': CHECKPOINT_DIR / f'{slug}.zip',
        'vecnorm': CHECKPOINT_DIR / f'{slug}_vecnormalize.pkl',
        'metrics': CHECKPOINT_DIR / f'{slug}_metrics.pt',
    }


def smooth_xy(x, y, window=20):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    if len(x) == 0 or len(y) == 0:
        return np.array([], dtype=np.float32), np.array([], dtype=np.float32)
    if len(y) < window:
        return x, y
    kernel = np.ones(window, dtype=np.float32) / window
    return x[window - 1:], np.convolve(y, kernel, mode='valid')


def plot_step_curve(ax, timesteps, rewards, *, color, label, window=20):
    x_raw = np.asarray(timesteps, dtype=np.float32)
    y_raw = np.asarray(rewards, dtype=np.float32)
    if len(x_raw) == 0:
        return
    ax.plot(x_raw, y_raw, color=color, alpha=0.12)
    x_smooth, y_smooth = smooth_xy(x_raw, y_raw, window=window)
    ax.plot(x_smooth, y_smooth, color=color, linewidth=2.5, label=label)


def evaluate_humanoid_policy(policy_fn, *, seed=123, max_steps=1000):
    env = make_humanoid_env(seed=seed)
    obs, _ = env.reset(seed=seed)
    x_start = float(env.unwrapped.data.qpos[0])
    total_reward = 0.0
    steps = 0

    while steps < max_steps:
        action = np.asarray(policy_fn(obs), dtype=np.float32)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += float(reward)
        steps += 1
        if terminated or truncated:
            break

    x_end = float(env.unwrapped.data.qpos[0])
    env.close()
    return {
        'distance': x_end - x_start,
        'reward': total_reward,
        'steps': steps,
    }


def render_humanoid_policy(policy_fn, *, seed=123, duration_seconds=10.0, max_steps=None):
    env = make_humanoid_env(seed=seed, render_mode='rgb_array')
    obs, _ = env.reset(seed=seed)
    x_start = float(env.unwrapped.data.qpos[0])
    total_reward = 0.0
    steps = 0
    frames = []
    if max_steps is None:
        dt = float(getattr(env.unwrapped, 'dt', 0.015))
        max_steps = int(round(duration_seconds / dt))

    while steps < max_steps:
        frame = env.render()
        if frame is not None:
            frames.append(frame.copy())
        action = np.asarray(policy_fn(obs), dtype=np.float32)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += float(reward)
        steps += 1
        if terminated or truncated:
            break

    x_end = float(env.unwrapped.data.qpos[0])
    env.close()
    return frames, {
        'distance': x_end - x_start,
        'reward': total_reward,
        'steps': steps,
    }


def train_humanoid_reinforce_baseline(*, step_budget=HUMANOID_BASELINE_STEP_BUDGET, gamma=0.99,
                                      lr_actor=3e-4, lr_critic=1e-3, hidden=128,
                                      init_log_std=-1.0, max_grad_norm=0.5, seed=0,
                                      checkpoint_label=None, checkpoint_meta=None, verbose=True):
    env = make_humanoid_env(seed=seed)
    obs0, _ = env.reset(seed=seed)
    obs_dim = int(obs0.shape[0])
    act_dim = int(env.action_space.shape[0])
    action_scale = float(env.action_space.high[0])

    loaded = maybe_load_humanoid_pg_checkpoint(
        kind='humanoid_reinforce_baseline',
        obs_dim=obs_dim,
        act_dim=act_dim,
        hidden=hidden,
        action_scale=action_scale,
        label=checkpoint_label or 'Humanoid REINFORCE + baseline',
        extra_meta=checkpoint_meta,
        with_critic=True,
    )
    if loaded is not None:
        env.close()
        actor, critic, rewards_history, extras = loaded
        metrics = {
            'episode_rewards': list(rewards_history),
            'timesteps': list(extras.get('timesteps', [])),
            'episode_lengths': list(extras.get('episode_lengths', [])),
            'episode_distances': list(extras.get('episode_distances', [])),
            'elapsed': float(extras.get('elapsed', 0.0)),
        }
        return actor, critic, metrics

    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    actor = ScaledGaussianPolicy(
        obs_dim,
        act_dim,
        hidden,
        init_log_std=init_log_std,
        action_scale=action_scale,
    ).to(device)
    critic = ValueNetwork(obs_dim, hidden).to(device)
    actor_opt = optim.Adam(actor.parameters(), lr=lr_actor)
    critic_opt = optim.Adam(critic.parameters(), lr=lr_critic)

    episode_rewards = []
    episode_lengths = []
    episode_distances = []
    timesteps = []
    steps_total = 0
    episode_idx = 0
    t0 = time.time()

    while steps_total < step_budget:
        obs, _ = env.reset(seed=seed + episode_idx)
        x_start = float(env.unwrapped.data.qpos[0])
        log_probs = []
        values = []
        rewards = []
        done = False
        truncated = False
        episode_steps = 0

        while not (done or truncated):
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            value = critic(obs_t)
            action_t, log_prob_t = actor.get_action(obs_t)
            obs, reward, done, truncated, _ = env.step(action_t.squeeze(0).detach().cpu().numpy())
            log_probs.append(log_prob_t.squeeze())
            values.append(value.squeeze())
            rewards.append(float(reward))
            episode_steps += 1

        returns_t = torch.as_tensor(discounted_returns(rewards, gamma=gamma), dtype=torch.float32, device=device)
        values_t = torch.stack(values)
        advantages_t = returns_t - values_t.detach()
        if len(advantages_t) > 1:
            advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        actor_loss = -(torch.stack(log_probs) * advantages_t).mean()
        critic_loss = nn.functional.mse_loss(values_t, returns_t)

        actor_opt.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
        actor_opt.step()

        critic_opt.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
        critic_opt.step()

        steps_total += episode_steps
        episode_idx += 1
        episode_rewards.append(float(np.sum(rewards)))
        episode_lengths.append(int(episode_steps))
        episode_distances.append(float(env.unwrapped.data.qpos[0] - x_start))
        timesteps.append(int(steps_total))

        if verbose and episode_idx % 25 == 0:
            tail = episode_rewards[-25:]
            print(
                f'  REINFORCE + baseline | episode {episode_idx:4d} | '
                f'steps {steps_total:6d} | avg reward {np.mean(tail):7.1f}'
            )

    elapsed = time.time() - t0
    env.close()

    if checkpoint_label is not None:
        meta = {
            'obs_dim': obs_dim,
            'act_dim': act_dim,
            'hidden': int(hidden),
            'critic_hidden': int(hidden),
            'action_scale': float(action_scale),
            'init_log_std': float(init_log_std),
        }
        if checkpoint_meta:
            meta.update(checkpoint_meta)
        save_pg_checkpoint({
            'kind': 'humanoid_reinforce_baseline',
            'label': checkpoint_label,
            'meta': meta,
            'policy_state_dict': actor.state_dict(),
            'critic_state_dict': critic.state_dict(),
            'rewards': np.asarray(episode_rewards, dtype=np.float32),
            'extras': {
                'timesteps': np.asarray(timesteps, dtype=np.int32),
                'episode_lengths': np.asarray(episode_lengths, dtype=np.int32),
                'episode_distances': np.asarray(episode_distances, dtype=np.float32),
                'elapsed': float(elapsed),
            },
        })

    metrics = {
        'episode_rewards': episode_rewards,
        'timesteps': timesteps,
        'episode_lengths': episode_lengths,
        'episode_distances': episode_distances,
        'elapsed': float(elapsed),
    }
    return actor, critic, metrics


def train_humanoid_reinforce_time_baseline(*, step_budget=HUMANOID_BASELINE_STEP_BUDGET, gamma=0.99,
                                           lr=3e-4, hidden=128, init_log_std=-1.0,
                                           max_grad_norm=0.5, seed=0,
                                           checkpoint_label=None, checkpoint_meta=None, verbose=True):
    env = make_humanoid_env(seed=seed)
    obs0, _ = env.reset(seed=seed)
    obs_dim = int(obs0.shape[0])
    act_dim = int(env.action_space.shape[0])
    action_scale = float(env.action_space.high[0])

    loaded = maybe_load_humanoid_pg_checkpoint(
        kind='humanoid_reinforce_time_baseline',
        obs_dim=obs_dim,
        act_dim=act_dim,
        hidden=hidden,
        action_scale=action_scale,
        label=checkpoint_label or 'Humanoid REINFORCE + time baseline',
        extra_meta=checkpoint_meta,
        with_critic=False,
    )
    if loaded is not None:
        env.close()
        actor, rewards_history, extras = loaded
        metrics = {
            'episode_rewards': list(rewards_history),
            'timesteps': list(extras.get('timesteps', [])),
            'episode_lengths': list(extras.get('episode_lengths', [])),
            'episode_distances': list(extras.get('episode_distances', [])),
            'elapsed': float(extras.get('elapsed', 0.0)),
        }
        return actor, metrics

    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    actor = ScaledGaussianPolicy(
        obs_dim,
        act_dim,
        hidden,
        init_log_std=init_log_std,
        action_scale=action_scale,
    ).to(device)
    optimizer = optim.Adam(actor.parameters(), lr=lr)

    time_baseline_sum = []
    time_baseline_count = []
    episode_rewards = []
    episode_lengths = []
    episode_distances = []
    timesteps = []
    steps_total = 0
    episode_idx = 0
    t0 = time.time()

    while steps_total < step_budget:
        obs, _ = env.reset(seed=seed + episode_idx)
        x_start = float(env.unwrapped.data.qpos[0])
        log_probs = []
        rewards = []
        done = False
        truncated = False
        episode_steps = 0

        while not (done or truncated):
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            action_t, log_prob_t = actor.get_action(obs_t)
            obs, reward, done, truncated, _ = env.step(action_t.squeeze(0).detach().cpu().numpy())
            log_probs.append(log_prob_t.squeeze())
            rewards.append(float(reward))
            episode_steps += 1

        returns_np = discounted_returns(rewards, gamma=gamma)
        baseline_np = np.zeros_like(returns_np)
        for t in range(len(returns_np)):
            if t < len(time_baseline_sum) and time_baseline_count[t] > 0:
                baseline_np[t] = time_baseline_sum[t] / time_baseline_count[t]

        advantages_t = torch.as_tensor(returns_np - baseline_np, dtype=torch.float32, device=device)
        if len(advantages_t) > 1:
            advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        actor_loss = -(torch.stack(log_probs) * advantages_t).mean()
        optimizer.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
        optimizer.step()

        while len(time_baseline_sum) < len(returns_np):
            time_baseline_sum.append(0.0)
            time_baseline_count.append(0)
        for t, G in enumerate(returns_np):
            time_baseline_sum[t] += float(G)
            time_baseline_count[t] += 1

        steps_total += episode_steps
        episode_idx += 1
        episode_rewards.append(float(np.sum(rewards)))
        episode_lengths.append(int(episode_steps))
        episode_distances.append(float(env.unwrapped.data.qpos[0] - x_start))
        timesteps.append(int(steps_total))

        if verbose and episode_idx % 25 == 0:
            tail = episode_rewards[-25:]
            print(
                f'  REINFORCE + time baseline | episode {episode_idx:4d} | '
                f'steps {steps_total:6d} | avg reward {np.mean(tail):7.1f}'
            )

    elapsed = time.time() - t0
    env.close()

    if checkpoint_label is not None:
        meta = {
            'obs_dim': obs_dim,
            'act_dim': act_dim,
            'hidden': int(hidden),
            'action_scale': float(action_scale),
            'init_log_std': float(init_log_std),
        }
        if checkpoint_meta:
            meta.update(checkpoint_meta)
        save_pg_checkpoint({
            'kind': 'humanoid_reinforce_time_baseline',
            'label': checkpoint_label,
            'meta': meta,
            'policy_state_dict': actor.state_dict(),
            'rewards': np.asarray(episode_rewards, dtype=np.float32),
            'extras': {
                'timesteps': np.asarray(timesteps, dtype=np.int32),
                'episode_lengths': np.asarray(episode_lengths, dtype=np.int32),
                'episode_distances': np.asarray(episode_distances, dtype=np.float32),
                'elapsed': float(elapsed),
            },
        })

    metrics = {
        'episode_rewards': episode_rewards,
        'timesteps': timesteps,
        'episode_lengths': episode_lengths,
        'episode_distances': episode_distances,
        'elapsed': float(elapsed),
    }
    return actor, metrics


def train_humanoid_ppo(*, total_timesteps=HUMANOID_PPO_STEP_BUDGET, seed=0, n_envs=4, verbose=True):
    meta = humanoid_ppo_meta(seed=seed, total_timesteps=total_timesteps, n_envs=n_envs)
    paths = humanoid_ppo_paths(meta)

    if not FORCE_RETRAIN and all(path.exists() for path in paths.values()):
        vec_env = make_humanoid_vec_env(n_envs=n_envs, seed=seed)
        vec_env = VecNormalize.load(str(paths['vecnorm']), vec_env)
        vec_env.training = False
        vec_env.norm_reward = False
        model = SB3PPO.load(str(paths['model']), env=vec_env, device=str(device))
        metrics = torch.load(paths['metrics'], map_location='cpu', weights_only=False)
        for key in ('episode_rewards', 'episode_lengths', 'timesteps', 'episode_distances'):
            metrics[key] = list(metrics.get(key, []))
        metrics['elapsed'] = float(metrics.get('elapsed', 0.0))
        print('Loaded checkpoint for Humanoid PPO; skipping retraining.')
        print(f"  path: {paths['model']}")
        return model, vec_env, metrics

    vec_env = make_humanoid_vec_env(n_envs=n_envs, seed=seed)
    callback = EpisodeStatsCallback()
    model = SB3PPO(
        'MlpPolicy',
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=seed,
        verbose=0,
        device=str(device),
        policy_kwargs={'net_arch': {'pi': [256, 256], 'vf': [256, 256]}},
    )

    t0 = time.time()
    model.learn(total_timesteps=total_timesteps, callback=callback, progress_bar=False)
    elapsed = time.time() - t0

    model.save(str(paths['model']))
    vec_env.save(str(paths['vecnorm']))
    metrics = {
        'label': 'Humanoid PPO',
        'meta': meta,
        'episode_rewards': list(callback.episode_rewards),
        'episode_lengths': list(callback.episode_lengths),
        'timesteps': list(callback.timesteps),
        'episode_distances': list(callback.episode_distances),
        'elapsed': float(elapsed),
    }
    torch.save(metrics, paths['metrics'])
    print(f"Saved Humanoid PPO checkpoint -> {paths['model']}")
    print(f"Saved VecNormalize stats -> {paths['vecnorm']}")

    vec_env.training = False
    vec_env.norm_reward = False
    return model, vec_env, metrics


def make_humanoid_actor_policy(actor):
    def policy_fn(obs):
        with torch.no_grad():
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            mu, _ = actor(obs_t)
            return mu.squeeze(0).cpu().numpy()

    return policy_fn


def make_humanoid_ppo_policy(model, vec_env):
    def policy_fn(obs):
        obs_batch = np.asarray(obs, dtype=np.float32)[None, :]
        obs_norm = vec_env.normalize_obs(obs_batch.copy())
        action, _ = model.predict(obs_norm, deterministic=True)
        return action[0]

    return policy_fn


def summarize_single_run(metrics, tail=50):
    rewards = np.asarray(metrics['episode_rewards'], dtype=np.float32)
    if len(rewards) == 0:
        return 0.0
    return float(np.mean(rewards[-min(tail, len(rewards)):]))


In [ ]:
# ============================================================
# Humanoid 3D: REINFORCE baselines vs PPO
# ============================================================

print('=== Humanoid 3D scaling test ===')
print('Task: Humanoid-v5')
humanoid_spec_env = gym.make(HUMANOID_ENV_ID)
print(f'Observation dim: {humanoid_spec_env.observation_space.shape[0]}')
print(f'Action dim: {humanoid_spec_env.action_space.shape[0]}')
humanoid_spec_env.close()
print(f'REINFORCE + baseline budget: {HUMANOID_BASELINE_STEP_BUDGET:,} environment steps')
print(f'PPO budget                 : {HUMANOID_PPO_STEP_BUDGET:,} environment steps')

humanoid_rb_label = 'Humanoid: REINFORCE + baseline'
humanoid_rb_meta = humanoid_pg_meta('humanoid_reinforce_baseline', seed=HUMANOID_SEED, step_budget=HUMANOID_BASELINE_STEP_BUDGET)
humanoid_actor_rb, humanoid_critic_rb, humanoid_rb_metrics = train_humanoid_reinforce_baseline(
    step_budget=HUMANOID_BASELINE_STEP_BUDGET,
    seed=HUMANOID_SEED,
    hidden=128,
    lr_actor=3e-4,
    lr_critic=1e-3,
    init_log_std=-1.0,
    checkpoint_label=humanoid_rb_label,
    checkpoint_meta=humanoid_rb_meta,
    verbose=True,
)

humanoid_tb_label = 'Humanoid: REINFORCE + time baseline'
humanoid_tb_meta = humanoid_pg_meta('humanoid_reinforce_time_baseline', seed=HUMANOID_SEED, step_budget=HUMANOID_BASELINE_STEP_BUDGET)
humanoid_actor_tb, humanoid_tb_metrics = train_humanoid_reinforce_time_baseline(
    step_budget=HUMANOID_BASELINE_STEP_BUDGET,
    seed=HUMANOID_SEED,
    hidden=128,
    lr=3e-4,
    init_log_std=-1.0,
    checkpoint_label=humanoid_tb_label,
    checkpoint_meta=humanoid_tb_meta,
    verbose=True,
)

humanoid_ppo_model, humanoid_ppo_vecenv, humanoid_ppo_metrics = train_humanoid_ppo(
    total_timesteps=HUMANOID_PPO_STEP_BUDGET,
    seed=HUMANOID_SEED,
    n_envs=4,
    verbose=True,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_step_curve(
    axes[0],
    humanoid_rb_metrics['timesteps'],
    humanoid_rb_metrics['episode_rewards'],
    color='tab:orange',
    label='REINFORCE + learned baseline',
    window=20,
)
plot_step_curve(
    axes[0],
    humanoid_tb_metrics['timesteps'],
    humanoid_tb_metrics['episode_rewards'],
    color='tab:purple',
    label='REINFORCE + time baseline',
    window=20,
)
plot_step_curve(
    axes[0],
    humanoid_ppo_metrics['timesteps'],
    humanoid_ppo_metrics['episode_rewards'],
    color='tab:red',
    label='PPO (4 envs + VecNormalize)',
    window=20,
)
axes[0].set_title('Humanoid-v5 reward vs environment steps', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Environment steps')
axes[0].set_ylabel('Episode reward')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

plot_step_curve(
    axes[1],
    humanoid_rb_metrics['timesteps'],
    humanoid_rb_metrics['episode_distances'],
    color='tab:orange',
    label='REINFORCE + learned baseline',
    window=20,
)
plot_step_curve(
    axes[1],
    humanoid_tb_metrics['timesteps'],
    humanoid_tb_metrics['episode_distances'],
    color='tab:purple',
    label='REINFORCE + time baseline',
    window=20,
)
plot_step_curve(
    axes[1],
    humanoid_ppo_metrics['timesteps'],
    humanoid_ppo_metrics['episode_distances'],
    color='tab:red',
    label='PPO (4 envs + VecNormalize)',
    window=20,
)
axes[1].set_title('Humanoid-v5 forward distance vs environment steps', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Environment steps')
axes[1].set_ylabel('Episode distance (m)')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)
plt.tight_layout()
plt.show()

humanoid_rb_last50 = summarize_single_run(humanoid_rb_metrics, tail=50)
humanoid_tb_last50 = summarize_single_run(humanoid_tb_metrics, tail=50)
humanoid_ppo_last50 = summarize_single_run(humanoid_ppo_metrics, tail=50)
humanoid_rb_dist50 = float(np.mean(np.asarray(humanoid_rb_metrics['episode_distances'], dtype=np.float32)[-50:]))
humanoid_tb_dist50 = float(np.mean(np.asarray(humanoid_tb_metrics['episode_distances'], dtype=np.float32)[-50:]))
humanoid_ppo_dist50 = float(np.mean(np.asarray(humanoid_ppo_metrics['episode_distances'], dtype=np.float32)[-50:]))
print('Final-window training reward:')
print(f'  REINFORCE + learned baseline: {humanoid_rb_last50:7.1f}')
print(f'  REINFORCE + time baseline   : {humanoid_tb_last50:7.1f}')
print(f'  PPO                         : {humanoid_ppo_last50:7.1f}')
print('Final-window forward distance:')
print(f'  REINFORCE + learned baseline: {humanoid_rb_dist50:7.2f} m')
print(f'  REINFORCE + time baseline   : {humanoid_tb_dist50:7.2f} m')
print(f'  PPO                         : {humanoid_ppo_dist50:7.2f} m')

humanoid_rb_policy = make_humanoid_actor_policy(humanoid_actor_rb)
humanoid_tb_policy = make_humanoid_actor_policy(humanoid_actor_tb)
humanoid_ppo_policy = make_humanoid_ppo_policy(humanoid_ppo_model, humanoid_ppo_vecenv)
humanoid_eval_rb = evaluate_humanoid_policy(humanoid_rb_policy, seed=123)
humanoid_eval_tb = evaluate_humanoid_policy(humanoid_tb_policy, seed=123)
humanoid_eval_ppo = evaluate_humanoid_policy(humanoid_ppo_policy, seed=123)

humanoid_eval_results = {
    'REINFORCE + learned baseline': humanoid_eval_rb,
    'REINFORCE + time baseline': humanoid_eval_tb,
    'PPO': humanoid_eval_ppo,
}

labels = list(humanoid_eval_results.keys())
distances = [humanoid_eval_results[label]['distance'] for label in labels]
colors = ['tab:orange', 'tab:purple', 'tab:red']
fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.bar(labels, distances, color=colors, edgecolor='white')
ax.set_ylabel('Forward distance (m)')
ax.set_title('Humanoid-v5 evaluation on a shared rollout seed', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
for bar, label in zip(bars, labels):
    result = humanoid_eval_results[label]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{result['distance']:.2f}m\nR={result['reward']:.1f}",
        ha='center',
        va='bottom',
        fontsize=10,
    )
plt.tight_layout()
plt.show()

print('Shared-seed evaluation:')
for label, result in humanoid_eval_results.items():
    print(
        f"{label:24s} | distance: {result['distance']:6.2f} m | "
        f"reward: {result['reward']:7.1f} | steps: {result['steps']:4d}"
    )


In [68]:
# Humanoid rollout videos for the trained policies
humanoid_rb_frames, humanoid_rb_video_result = render_humanoid_policy(
    humanoid_rb_policy,
    seed=12,
    duration_seconds=10.0,
)
humanoid_tb_frames, humanoid_tb_video_result = render_humanoid_policy(
    humanoid_tb_policy,
    seed=12,
    duration_seconds=10.0,
)
humanoid_ppo_frames, humanoid_ppo_video_result = render_humanoid_policy(
    humanoid_ppo_policy,
    seed=12,
    duration_seconds=10.0,
)

humanoid_rb_video = show_video(
    humanoid_rb_frames,
    fps=30,
    title=(
        f"Humanoid REINFORCE + baseline — "
        f"{humanoid_rb_video_result['distance']:.2f}m, R={humanoid_rb_video_result['reward']:.1f}"
    ),
).data
humanoid_tb_video = show_video(
    humanoid_tb_frames,
    fps=30,
    title=(
        f"Humanoid REINFORCE + time baseline — "
        f"{humanoid_tb_video_result['distance']:.2f}m, R={humanoid_tb_video_result['reward']:.1f}"
    ),
).data
humanoid_ppo_video = show_video(
    humanoid_ppo_frames,
    fps=30,
    title=(
        f"Humanoid PPO — "
        f"{humanoid_ppo_video_result['distance']:.2f}m, R={humanoid_ppo_video_result['reward']:.1f}"
    ),
).data

display(HTML(
    f"""
<div style='display:flex;flex-direction:column;gap:18px;align-items:stretch;'>
  <div style='background:#fff;border:1px solid #ddd;border-radius:10px;padding:12px;'>
    <div style='font-weight:700;font-size:16px;margin-bottom:8px;'>Humanoid REINFORCE + learned baseline</div>
    {humanoid_rb_video}
  </div>
  <div style='background:#fff;border:1px solid #ddd;border-radius:10px;padding:12px;'>
    <div style='font-weight:700;font-size:16px;margin-bottom:8px;'>Humanoid REINFORCE + time baseline</div>
    {humanoid_tb_video}
  </div>
  <div style='background:#fff;border:1px solid #ddd;border-radius:10px;padding:12px;'>
    <div style='font-weight:700;font-size:16px;margin-bottom:8px;'>Humanoid PPO</div>
    {humanoid_ppo_video}
  </div>
</div>
"""
))


## Reading the result

The quantity on the right is not a formal variance proof. It is a practical proxy: the standard deviation of the **raw advantage weights** used before normalization.

What the current saved output actually shows:
- The current PPO tuning sweep is only a **1-seed quick sweep**, so the PPO config ranking itself is still provisional.
- Within that saved sweep, the best PPO preset is `batch8_clip0.2_epochs8_lr1e-3_1e-3_std-1.0`, reaching about **24.9** average reward over the last 50 episodes and about **0.48 m** in the 10-second rollout.
- On this crawler task, PPO is **not** the top performer in raw reward: the saved comparison reports about **35.1** for the time-dependent baseline, **33.3** for the constant baseline, **31.5** for Actor-Critic, **29.7** for vanilla REINFORCE, and **24.9** for PPO.
- PPO's strongest evidence here is the variance proxy: its raw advantage std is about **0.57**, versus **1.28** for the time-dependent baseline and roughly **1.78 to 2.01** for the other policy-gradient baselines. So on the crawler, PPO currently looks more like a **variance-control method** than a raw-performance win.

Additional learning objective: scaling beyond the crawler
- The crawler is still a small 2D task with only 2 continuous actions. That is useful for understanding the algorithm, but it is not yet the setting where PPO should be expected to show its biggest practical advantage.
- The Humanoid section makes that scaling point concrete with three methods: `REINFORCE + learned baseline`, `REINFORCE + time-dependent baseline`, and `PPO`. The two REINFORCE-style baselines are run for **100k** environment steps, while `PPO` is allowed to continue to **300k** steps on the same task.
- On Humanoid, total reward includes a large survival term, so the cleaner locomotion metric is **forward distance**. The notebook therefore emphasizes both reward and forward-distance curves, plus the shared-seed rollout videos.
- The actual saved Humanoid results separate the baselines clearly. By the last 50 episodes, the learned baseline reaches about **0.20 m** per episode, the time-dependent baseline is already failing at about **-0.05 m**, and PPO reaches about **0.41 m**.
- The shared-seed rollouts tell the same story: the learned baseline travels about **0.46 m**, the time-dependent baseline falls back to about **-0.09 m**, and PPO reaches about **0.59 m**.
- So the intended lesson now has direct empirical support in this notebook: once the task becomes more complex, a weaker REINFORCE baseline can collapse, a learned baseline helps but still plateaus sooner, and PPO can keep learning and scale more gracefully when training continues.
- In other words, the crawler notebook motivates the mechanism, and the humanoid demo supplies the stronger scaling evidence.